# TasteAI: Movie Recommendation System
### Content-Based Filtering with CountVectorizer & Cosine Similarity
This notebook demonstrates the end-to-end preprocessing, feature extraction (Director, Cast, Keywords, Genres, Overview), text stemming, vectorization, and recommendation querying.

In [ ]:
import os
import pickle
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from ml.preprocessing.movie_preprocessing import preprocess_movies

In [ ]:
# 1. Load and Preprocess Datasets
movies_path = '../../datasets/movies/tmdb_5000_movies.csv'
credits_path = '../../datasets/movies/tmdb_5000_credits.csv'
df = preprocess_movies(movies_path, credits_path)
df.head(3)

In [ ]:
# 2. Vectorize Tags
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(df['tags']).toarray()
print('Vocabulary size:', len(cv.get_feature_names_out()))
print('Vectors shape:', vectors.shape)

In [ ]:
# 3. Compute Cosine Similarity Matrix
similarity = cosine_similarity(vectors)
similarity.shape

In [ ]:
# 4. Recommendation Function
def recommend(movie_title, top_k=5):
    if movie_title not in df['title'].values:
        return f'Movie {movie_title} not found in catalog.'
    idx = df[df['title'] == movie_title].index[0]
    distances = sorted(list(enumerate(similarity[idx])), reverse=True, key=lambda x: x[1])
    
    print(f'Top {top_k} recommendations for {movie_title}:')
    for rank, (m_idx, sim) in enumerate(distances[1:top_k+1], 1):
        m = df.iloc[m_idx]
        print(f'{rank}. {m["title"]} ({m.get("release_date", "")[:4]}) | Director: {m["director"]} | Similarity: {sim:.3f}')

recommend('Avatar')